# Train / Validation / Test Splits

<hr style="height: 5px; background-color: #3CB371 ; border: none;">

### Introduction:
Previously, we split our data into two parts: a training set and a test set. The training set was used to train the model, allowing it to learn patterns from the data, while the test set was kept aside to evaluate how well the trained model performs on unseen data.

However, there is a problem with relying on only these two sets when we start tuning our model. If we repeatedly use the test set to compare different models or try different hyperparameters, the test results start influencing our decisions. Over time, we may end up choosing a model that performs well specifically on that test set, rather than one that truly generalizes to new, unseen data.

In this noteboook, we are introducing the **Validation set**. A new separate set that wil help us during the development and tuning process.

<hr style="height: 5px; background-color: #3CB371 ; border: none;">

# Section 1 : Concepts

> A train, validation, test split divides your dataset into three parts: a training set the model learns from, a validation set for tuning and monitoring during training, and a test set held back until the end for an unbiased measure of how the model will perform in production. The split exists to prevent overfitting and to keep evaluation honest.

>  <div style="text-align: center;"> <img src="intro.png" alt="A descriptive summary of the photo" width="500" > </div>

### Common Split Ratios


| Dataset Size | Train % | Validation % | Test % | Best Use Case |
| :--- | :---: | :---: | :---: | :--- |
| **Small / Medium** (< 100,000 samples) | 70% | 15% | 15% | Standard baseline split |
| **Standard** | 80% | 10% | 10% | Most common distribution |
| **Large-scale** (Millions of samples) | 98% | 1% | 1% | Deep learning / LLMs where 1% yields thousands of testing samples |

 #### Note:
>  we will use the (60% / 20% / 20%) in this notebook

-----

## Why the Validation set ?

> The validation set is introduced to provide unseen data for model selection and hyperparameter tuning, while keeping the test set independent so it can provide an unbiased estimate of the final model's generalization performance

### 1. Model Selection and tuning

The validation set gives us a separate dataset that we can use during the model development process. It allows us to make decisions about our model without using the test set.
The validation set can help us with both:

- **Model selection:** Which algorithm should we use?
- **Hyperparameter tuning:** Which configuration of that algorithm should we use?

Model tuning is not limited to changing hyperparameters. We may also want to compare **different algorithms** and determine which one performs best for our problem.

For example, we might compare:

- Logistic Regression
- Decision Tree
- k-NN
- SVM

For each algorithm, we can also try different hyperparameter configurations.

For example, for a Decision Tree:

```python
DecisionTreeClassifier(max_depth=3)
DecisionTreeClassifier(max_depth=5)
DecisionTreeClassifier(max_depth=10)
```

Each configuration is trained using the **training set**, and its performance is then evaluated using the **validation set**.

For example:

| Model | Hyperparameter | Training Accuracy | Validation Accuracy |
|-------|----------------|-------------------|---------------------|
| Decision Tree | `max_depth=3` | 87% | 85% |
| Decision Tree | `max_depth=5` | 93% | **91%** |
| Decision Tree | `max_depth=10` | 98% | 88% |
| Logistic Regression | `C=1` | 90% | 89% |
| k-NN | `k=5` | 94% | 90% |

Based on the validation results, we can choose the **Decision Tree with `max_depth=5`**, because it achieved the highest validation accuracy.



The training set is used to **fit each candidate model**, while the validation set is used to **compare the candidates and make our selection**.

----


### 2. Detecting overfitting

The validation set also helps us identify when a model is **overfitting**.

> Overfitting occurs when a model learns the training data too closely, including patterns that do not generalize well to new data. As a result, the model performs very well on the training set but performs worse on unseen data.
>   <div style="text-align: center;"> <img src="Overfitting.png" alt="A descriptive summary of the photo" width="500" > </div>
-------
For example:

| Model | Training Accuracy | Validation Accuracy |
|-------|-------------------|---------------------|
| `max_depth=3` | 87% | 85% |
| `max_depth=5` | 93% | **91%** |
| `max_depth=10` | 98% | 88% |
| `max_depth=20` | 100% | 82% |

As the tree becomes deeper, its training accuracy continues to increase. However, its validation accuracy starts to decrease.

This gap is a warning sign that the model is becoming too complex and is **overfitting the training data**.

The validation set helps us respond to this by allowing us to choose a configuration that gives a better balance between **fitting the training data and generalizing to unseen data**.

For example, instead of choosing `max_depth=20` because it has 100% training accuracy, we may choose `max_depth=5` because it achieves the best validation performance.

#### Note

The validation set does not directly prevent overfitting. But it helps us **detect overfitting and make better decisions to reduce it**, such as:

- choosing a simpler model,
- reducing the model's complexity,
- changing hyperparameters,
- applying regularization.

The final **test set is then kept separate** and used only after these decisions have been made, providing a more reliable estimate of how well the final model generalizes to completely unseen data.

<hr style="height: 5px; background-color: #3CB371 ; border: none;">

# Section 2 : Code Implementation


#### The general process will be:

1. Split the dataset into **training, validation, and test sets**.
2. Train our initial models using the **training set**.
3. Evaluate the models on the **validation set**.
4. Tune the models by changing their hyperparameters and comparing their validation performance.
5. Select the best-performing model and hyperparameters.
6. Evaluate the final chosen models on the **test set** only at the end.

For this example, we will use the **Breast Cancer dataset** and compare **SVM** and **Decision Tree**, since both models previously achieved around 95% accuracy **(this was implemented in Week3-Day4 Notebook of the same repository)**.

The goal is to determine whether we can improve their performance through proper validation and hyperparameter tuning.

----------

### Step 1 : Split the dataset into training, validation, and test sets.

We first hold out **20% of the original data as the final test set**.

```python
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

dataset = load_breast_cancer()

X = dataset.data
y = dataset.target

# Hold out 20% as the final test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
```

 #### Where:

> - `X_test`, `y_test` → final test set (20%)
> - `X_temp`, `y_temp` → remaining 80%

We then split the remaining 80% into **training and validation sets**:

```python
# Split the remaining 80% into:
# 80% training and 20% validation

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.2,
    random_state=42,
    stratify=y_temp
)
```

This gives us approximately:

```text
Original Dataset
       │
       ├── Training      → 64%
       ├── Validation    → 16%
       └── Test          → 20%
```
-------


### Step 2 : Scale the Data

Because **SVM is sensitive to the scale of the features**, we should standardize the data before training it.

However, the scaler must be fitted **only on the training data**.

```python
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)
```

**Important rule :**

> **Fit the scaler only on the training set, then use that same fitted scaler to transform the validation and test sets.**

We should **not** do this:

```python
X_temp = scaler.fit_transform(X_temp)
```

This is a form of **data leakage**.
> because `X_temp` contains both the training and validation data. Fitting the scaler on it would allow information from the validation set to influence the preprocessing.



For the Decision Tree, scaling is generally unnecessary because tree-based models make decisions based on feature thresholds rather than distances or feature magnitudes. Therefore, we can use the original features for the Decision Tree.

---------


### Step 3 : Train the Initial Models

Now we train our two initial models using **only the training set**.

```python
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

svm_model = SVC()

tree_model = DecisionTreeClassifier(random_state=42)

svm_model.fit(X_train_scaled, y_train)

tree_model.fit(X_train, y_train)
```

At this stage, the models have learned from the training data, but we have not used the test set.

---


### Step 4 :  Evaluate the Models on the Validation Set

We now use the validation set to see how well each model performs on unseen data.

```python
from sklearn.metrics import accuracy_score

svm_val_predictions = svm_model.predict(X_val_scaled)

tree_val_predictions = tree_model.predict(X_val)

svm_val_accuracy = accuracy_score(y_val, svm_val_predictions)

tree_val_accuracy = accuracy_score(y_val, tree_val_predictions)

print("SVM Validation Accuracy:", svm_val_accuracy)
print("Decision Tree Validation Accuracy:", tree_val_accuracy)
```

The validation results give us a baseline for comparison.

For example:

```text
SVM Validation Accuracy: 95%
Decision Tree Validation Accuracy: 93%
```

We can now begin tuning the models.

---

### Step 5 : Tune the Models Using the Validation Set

In this example, we will experiment with the `C` and `kernel` hyperparameters:

```python
svm_1 = SVC(C=0.1, kernel="linear")
svm_2 = SVC(C=1, kernel="rbf")
svm_3 = SVC(C=10, kernel="rbf")

svm_1.fit(X_train_scaled, y_train)
svm_2.fit(X_train_scaled, y_train)
svm_3.fit(X_train_scaled, y_train)

print("SVM 1:", accuracy_score(y_val, svm_1.predict(X_val_scaled)))
print("SVM 2:", accuracy_score(y_val, svm_2.predict(X_val_scaled)))
print("SVM 3:", accuracy_score(y_val, svm_3.predict(X_val_scaled)))
```

We choose the configuration with the best **validation accuracy**, rather than the best training accuracy.

#### Decision Tree

For the Decision Tree, we can experiment with `max_depth`:

```python
tree_1 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_2 = DecisionTreeClassifier(max_depth=5, random_state=42)
tree_3 = DecisionTreeClassifier(max_depth=10, random_state=42)

tree_1.fit(X_train, y_train)
tree_2.fit(X_train, y_train)
tree_3.fit(X_train, y_train)

print("Tree 1:", accuracy_score(y_val, tree_1.predict(X_val)))
print("Tree 2:", accuracy_score(y_val, tree_2.predict(X_val)))
print("Tree 3:", accuracy_score(y_val, tree_3.predict(X_val)))
```

The validation results allow us to determine which configuration provides the best balance between learning the training data and generalizing to unseen data.

---

### Step 6 : Select the Best Configuration

After trying different models and hyperparameters, we select the configuration with the best validation performance.

For example:

```text
SVM (C=1, kernel='rbf')        → 97% validation accuracy
Decision Tree (max_depth=5)   → 94% validation accuracy
```

In this example, the SVM configuration would be selected as the best candidate.

At this point, we have finished using the validation set for model selection and tuning.

---



### 7. Evaluate the Final Model on the Test Set



Finally, we evaluate the selected model on the **test set**.

The test set has not been used for training, hyperparameter tuning, or model selection.

```python
best_svm = SVC(C=1, kernel="rbf")

best_svm.fit(X_train_scaled, y_train)

test_predictions = best_svm.predict(X_test_scaled)

test_accuracy = accuracy_score(y_test, test_predictions)

print("Final Test Accuracy:", test_accuracy)
```

The test accuracy gives us our final estimate of how well the selected model performs on completely unseen data.

> **Important:** We should not keep changing the model based on the test accuracy. If we do, the test set starts influencing our decisions, which defeats the purpose of keeping it as an independent final evaluation.


<hr style="height: 5px; background-color: #3CB371 ; border: none;">

# Section 3 : Hands-on Lap

In this section, we will put the concepts from the previous sections into practice by training and comparing five different classification models:

- **Logistic Regression**
- **Support Vector Machine (SVM)**
- **K-Nearest Neighbors (KNN)**
- **Decision Tree**
- **Random Forest**

For each model, we will tune **one hyperparameter** using the validation set:

| Model | Hyperparameter |
|-------|----------------|
| Logistic Regression | `C` |
| SVM | `C` |
| KNN | `n_neighbors` |
| Decision Tree | `max_depth` |
| Random Forest | `n_estimators` |

 **Accuracy** will be used as the main evaluation metric so that the performance of all five models can be compared using the same measure.

-----------

## Step 1: Take a Week 3 dataset and create a 60/20/20 train/validation/test split with a fixed random_state

> Using the **Breast Cancer dataset from scikit-learn**

In [3]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler

dataset = load_breast_cancer()

X = dataset.data
y = dataset.target

# 1) hold out 20% as the final test set
X_temp, X_test, y_temp, y_test = train_test_split(
 X, y, test_size=0.2, random_state=42)

# 2) split the rest into train (60%) and validation (20%)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.2,
    random_state=42,
    stratify=y_temp
)
scaler = StandardScaler()


X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

-----------

## Step 2: Train a model on the training set and tune one setting by checking the validation set only.

### 1. Importing All Five models and fitting them into the training set

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# Logistic Regression
logistic_model = LogisticRegression(max_iter=4000)
logistic_model.fit(X_train_scaled, y_train)

# SVM
svm_model = SVC()
svm_model.fit(X_train_scaled, y_train)

# K-NN
knn_model = KNeighborsClassifier()
knn_model.fit(X_train_scaled, y_train)

# Decision Tree
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

# Random Forest
forest_model = RandomForestClassifier(random_state=42)
forest_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### 2. Checking the validation accurecy for all models before any hyperparameters tuning

In [18]:
from sklearn.metrics import accuracy_score

logistic_val_pred = logistic_model.predict(X_val_scaled)
svm_val_pred = svm_model.predict(X_val_scaled)
knn_val_pred = knn_model.predict(X_val_scaled)

tree_val_pred = tree_model.predict(X_val)
forest_val_pred = forest_model.predict(X_val)

print("Logistic Regression validation accurecy:", accuracy_score(y_val, logistic_val_pred))
print("SVM validation accurecy:",accuracy_score(y_val, svm_val_pred))
print("KNN validation accurecy:",accuracy_score(y_val, knn_val_pred))
print("Decision Tree validation accurecy:",accuracy_score(y_val, tree_val_pred))
print("Random Forest validation accurecy:",accuracy_score(y_val, forest_val_pred))

Logistic Regression validation accurecy: 0.978021978021978
SVM validation accurecy: 0.989010989010989
KNN validation accurecy: 0.967032967032967
Decision Tree validation accurecy: 0.9010989010989011
Random Forest validation accurecy: 0.967032967032967


### Observation
> Here , SVM scored the highest validation accurecy. But we will still tune the rest of the algorithms to see if we can make any improvements

### 3. Tuning one hyperparameter for each model

#### `C` value for logetic Regression

In [29]:
C_values = [0.01, 0.1, 1, 10, 100]

for C in C_values:
    model = LogisticRegression(C=C, max_iter=4000)
    model.fit(X_train_scaled, y_train)

    predictions = model.predict(X_val_scaled)
    accuracy = accuracy_score(y_val, predictions)

    print("C =", C)
    print("Validation Accuracy =", accuracy)

C = 0.01
Validation Accuracy = 0.9340659340659341
C = 0.1
Validation Accuracy = 0.978021978021978
C = 1
Validation Accuracy = 0.978021978021978
C = 10
Validation Accuracy = 0.967032967032967
C = 100
Validation Accuracy = 0.945054945054945


> **The hyperparameter that resulted with the highest validation accurecy :** C = 0.1 and C = 1 , 

#### `C` for SVM

In [30]:
for C in C_values:
    model = SVC(C=C)
    model.fit(X_train_scaled, y_train)

    predictions = model.predict(X_val_scaled)
    accuracy = accuracy_score(y_val, predictions)

    print("C =", C)
    print("Validation Accuracy =", accuracy)

C = 0.01
Validation Accuracy = 0.6263736263736264
C = 0.1
Validation Accuracy = 0.945054945054945
C = 1
Validation Accuracy = 0.989010989010989
C = 10
Validation Accuracy = 0.967032967032967
C = 100
Validation Accuracy = 0.945054945054945


> **The hyperparameter that resulted with the highest validation accurecy :** C = 1 

####  `n_neighbors` for KNN 

In [20]:
k_values = [3, 5, 7, 9, 11]

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_scaled, y_train)

    predictions = model.predict(X_val_scaled)
    accuracy = accuracy_score(y_val, predictions)

    print(f"k = {k}, Validation Accuracy = {accuracy:.4f}")

k = 3, Validation Accuracy = 0.9560
k = 5, Validation Accuracy = 0.9670
k = 7, Validation Accuracy = 0.9780
k = 9, Validation Accuracy = 0.9780
k = 11, Validation Accuracy = 0.9670


> **The hyperparameter that resulted with the highest validation accurecy :** k = 7 and k = 9 , 

#### `max_depth` for Decision Tree 

In [21]:
depth_values = [3, 5, 7, 10, 15]

for depth in depth_values:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )
    model.fit(X_train, y_train)

    predictions = model.predict(X_val)
    accuracy = accuracy_score(y_val, predictions)

    print(f"max_depth = {depth}, Validation Accuracy = {accuracy:.4f}")

max_depth = 3, Validation Accuracy = 0.9231
max_depth = 5, Validation Accuracy = 0.9121
max_depth = 7, Validation Accuracy = 0.9011
max_depth = 10, Validation Accuracy = 0.9011
max_depth = 15, Validation Accuracy = 0.9011


> **The hyperparameter that resulted with the highest validation accurecy :** max_depth = 3

#### `n_estimators` for Random Forest 

In [22]:
n_values = [50, 100, 200, 300]

for n in n_values:
    model = RandomForestClassifier(
        n_estimators=n,
        random_state=42
    )
    model.fit(X_train, y_train)

    predictions = model.predict(X_val)
    accuracy = accuracy_score(y_val, predictions)

    print(f"n_estimators = {n}, Validation Accuracy = {accuracy:.4f}")

n_estimators = 50, Validation Accuracy = 0.9670
n_estimators = 100, Validation Accuracy = 0.9670
n_estimators = 200, Validation Accuracy = 0.9560
n_estimators = 300, Validation Accuracy = 0.9560


> **The hyperparameter that resulted with the highest validation accurecy :** n_estimators = 50 and n_estimators = 100

-----------------

## Step 3: Evaluate the final model on the test set exactly once and report the score.

### Note:
> The final Models will use the hyper parameters that game them the heigest validation accurecy

In [33]:
final_SVM_model = SVC(C=1)

final_SVM_model.fit(X_train_scaled, y_train)
test_predictions = final_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Final SVM Test Accuracy: {test_accuracy:.4f}")

print ("---------------------------")

final_logistic_model = LogisticRegression(C=1, max_iter=4000)

final_logistic_model.fit(X_train_scaled, y_train)
test_predictions = final_logistic_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Final Logistic Regression Test Accuracy: {test_accuracy:.4f}")

print ("---------------------------")

# Final KNN Model
final_KNN_model = KNeighborsClassifier(n_neighbors=7)

final_KNN_model.fit(X_train_scaled, y_train)

test_predictions = final_KNN_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Final KNN Test Accuracy: {test_accuracy:.4f}")

print ("---------------------------")

# Final Decision Tree Model
final_tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)

final_tree_model.fit(X_train, y_train)

test_predictions = final_tree_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Final Decision Tree Test Accuracy: {test_accuracy:.4f}")

print ("---------------------------")

# Final Random Forest Model
final_forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

final_forest_model.fit(X_train, y_train)

test_predictions = final_forest_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"Final Random Forest Test Accuracy: {test_accuracy:.4f}")

Final SVM Test Accuracy: 0.9649
---------------------------
Final Logistic Regression Test Accuracy: 0.9912
---------------------------
Final KNN Test Accuracy: 0.9649
---------------------------
Final Decision Tree Test Accuracy: 0.9298
---------------------------
Final Random Forest Test Accuracy: 0.9561


### Choosing the best model:

| Model | Validation Accuracy | Final Test Accuracy |
|---|---:|---:|
| Logistic Regression | 97.8% | **99.12%** |
| SVM | **98.9%** | 96.49% |
| KNN | 97.8% | 96.49% |
| Decision Tree | 92.31% | 92.98% |
| Random Forest | 96.7% | 95.61% |

### Final Observation :
**SVM** achieved the highest performance (98.9%) in validation accurecy and was therefore selected as the best model during the model-selection stage. When evaluated on the previously unseen test set, it achieved an accuracy of 96.49%. 

Although Logistic Regression  wasnt selected for the validation performence, it surprisingly achieved the highest test accuracy (99.12%)

---------

## Step 4: In Markdown, explain what would go wrong if you had tuned against the test set instead

> If we used the test set to tune the hyperparameters and the models, the data would no longer be truly unseen. Gradually, as we keep experimenting and making our decisions based on the test set, the models and their configurations would adapt to this specific test set and perform better and better on it.
> 
> This can lead to **overfitting to the test set**, where the model becomes too adapted to the patterns in this particular data. As a result, when facing another completely unseen dataset outside of the test set, it may perform poorly because our model-selection process has become too specific to the test data.
> 
> Therefore, we keep the test set untouched until the end so that it can give us a more reliable estimate of the model's real performance on unseen data.


<hr style="height: 5px; background-color: #3CB371 ; border: none;">

# References :

- Read more about overfitting : https://datahacker.rs/018-pytorch-popular-techniques-to-prevent-the-overfitting-in-a-neural-networks/
- All the codes in Section 2 were an implementaion from previous notebooks:
  - Day4 : https://github.com/thimarArda/BinX_AI_-_ML_Training/tree/main/Week3_Spervised_Learning/Day4_Trees_Forests_SVMs_kNN
  - Day5: https://github.com/thimarArda/BinX_AI_-_ML_Training/tree/main/Week3_Spervised_Learning/Day5_Full_ML_Pipeline
- Decesion Tree, SVM, K-NN, Random Forest algorithms and their hyperparameters :
 > https://github.com/thimarArda/BinX_AI_-_ML_Training/tree/main/Week3_Spervised_Learning/Day4_Trees_Forests_SVMs_kNN

